# 1. Setup & Imports
This notebook runs a Random Forest forecasting experiment for FinSight.

- Doc: loads data, preprocesses, runs CV, hyperparameter tuning, final training, and saves artifacts.
- Logging: prints concise progress and results at each major step.

In [1]:
import sys
sys.path.append("../../../")

import json
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor

from backend.data.data_loader import fetch_stock_data
from backend.data.preprocessor import FinSightPreprocessor

optuna.logging.set_verbosity(optuna.logging.INFO)

TICKER = "HDFCBANK.NS"
START = "2020-01-01"
END = "2026-05-01"
TEST_START = "2025-01-01"

safe_ticker = ''.join(ch if ch.isalnum() else '_' for ch in TICKER)
ARTIFACTS_DIR = Path("./")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ticker: {TICKER}")
print(f"Date range: {START} to {END}")
print(f"Artifacts dir: {ARTIFACTS_DIR.resolve()}")

c:\Users\gaura\Desktop\FinSight\finsightvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ticker: HDFCBANK.NS
Date range: 2020-01-01 to 2026-05-01
Artifacts dir: C:\Users\gaura\Desktop\FinSight\notebooks\models\random_forest


# 2. Load Data
Load OHLCV data using FinSight's shared data loader and inspect shape and schema.

In [2]:
df = fetch_stock_data(TICKER, START, END)
print("Shape:", df.shape)
display(df.head())
df.info()

Shape: (1567, 9)


Price,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread
Date,,,,,,,,,
2020-01-02,639.500000,644.000000,639.500000,643.375000,6137166,609.389160,0.006374,0.006354,4.500000
2020-01-03,641.099976,642.500000,631.799988,634.200012,10855550,600.698792,-0.014261,-0.014363,10.700012
2020-01-06,630.000000,630.900024,618.000000,620.474976,10890186,587.698792,-0.021641,-0.021879,12.900024
2020-01-07,629.450012,635.724976,626.125000,630.299988,14724494,597.004822,0.015835,0.015711,9.599976
2020-01-08,623.474976,631.075012,620.025024,628.650024,11332110,595.441956,-0.002618,-0.002621,11.049988


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1567 entries, 2020-01-02 to 2026-05-01
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          1567 non-null   float64
 1   High          1567 non-null   float64
 2   Low           1567 non-null   float64
 3   Close         1567 non-null   float64
 4   Volume        1567 non-null   int64  
 5   Adj Close     1567 non-null   float64
 6   daily_return  1567 non-null   float64
 7   log_return    1567 non-null   float64
 8   hl_spread     1567 non-null   float64
dtypes: float64(8), int64(1)
memory usage: 122.4 KB


# 3. Feature Engineering
Apply FinSight preprocessing and optionally enrich the feature frame using graph features if the JSON artifact exists.
The target variable is next-day Close (`Close.shift(-1)`).

In [3]:
preprocessor = FinSightPreprocessor(ticker=TICKER)
processed_df = preprocessor.fit_transform(df)

graph_features_path = ARTIFACTS_DIR / "graph_features.json"
if graph_features_path.exists():
    with graph_features_path.open("r", encoding="utf-8") as f:
        graph_features = json.load(f)
    for k, v in graph_features.items():
        processed_df[k] = float(v)
    print(f"Loaded graph features from {graph_features_path}")
else:
    graph_features = {}
    print(f"No graph features found at {graph_features_path}; continuing without them.")

model_df = processed_df.copy()
model_df["target"] = model_df["Close"].shift(-1)
model_df = model_df.dropna(subset=["target"]).replace([np.inf, -np.inf], np.nan).dropna()

print("Final model frame shape:", model_df.shape)
display(model_df.head())
print("Feature columns (excluding target):", len([c for c in model_df.columns if c != "target"]))

No graph features found at graph_features.json; continuing without them.
Final model frame shape: (1547, 15)


,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread,Close_lag_1,Close_lag_5,Close_lag_10,rolling_mean_20,rolling_std_20,target
Date,,,,,,,,,,,,,,,
2020-01-29,-1.277790,-1.271400,-1.224250,-1.233929,-0.545518,-1.240527,0.617558,0.620802,-0.608115,-1.282271,-1.207101,-1.030540,-1.141682,-0.666851,-1.272539
2020-01-30,-1.223918,-1.283469,-1.244035,-1.272539,-0.617486,-1.275065,-0.504253,-0.496689,-0.470182,-1.232459,-1.191381,-1.017203,-1.153743,-0.654015,-1.271554
2020-01-31,-1.253518,-1.288019,-1.232086,-1.271554,-0.657571,-1.274184,-0.004874,0.003287,-0.759839,-1.271048,-1.192560,-1.054468,-1.162108,-0.614829,-1.403536
2020-02-03,-1.389483,-1.445708,-1.398789,-1.403536,-0.576056,-1.392247,-1.694632,-1.705219,-0.573632,-1.270064,-1.315766,-1.145670,-1.171675,-0.434574,-1.257765
2020-02-04,-1.385536,-1.303056,-1.319257,-1.257765,-0.240307,-1.261849,1.887106,1.861340,0.512595,-1.401976,-1.276466,-1.187054,-1.177795,-0.416229,-1.199259


Feature columns (excluding target): 14


# 4. Train/Test Split
Create a chronological split using the date-based `TEST_START`.

In [4]:
train_df = model_df[model_df.index < "2025-01-01"]
test_df = model_df[model_df.index >= "2025-01-01"]

feature_cols = [c for c in model_df.columns if c != "target"]
X_train, y_train = train_df[feature_cols], train_df["target"]
X_test, y_test = test_df[feature_cols], test_df["target"]

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Features:", len(feature_cols))

Train size: 1218
Test size: 329
Features: 14


# 5. Time Series Cross Validation
Run baseline cross-validation using `TimeSeriesSplit(n_splits=5)` and report fold RMSE.

In [5]:
tscv = TimeSeriesSplit(n_splits=5)
baseline_rmses = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_tr, y_tr)

    preds = model.predict(X_val)
    rmse = float(np.sqrt(mean_squared_error(y_val, preds)))
    baseline_rmses.append(rmse)
    print(f"Fold {fold} RMSE: {rmse:.6f}")

print(f"Mean CV RMSE: {np.mean(baseline_rmses):.6f}")

Fold 1 RMSE: 0.455382
Fold 2 RMSE: 0.165259
Fold 3 RMSE: 0.085359
Fold 4 RMSE: 0.086464
Fold 5 RMSE: 0.244066
Mean CV RMSE: 0.207306


# 6. Hyperparameter Tuning (Optuna)
Tune Random Forest hyperparameters with 20 Optuna trials using time-series CV.

In [6]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state": 42,
        "n_jobs": -1
    }

    cv = TimeSeriesSplit(n_splits=3)
    rmses = []
    for tr_idx, val_idx in cv.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = RandomForestRegressor(**params)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
        rmses.append(rmse)

    print(f"Trial {trial.number} | RMSE: {float(np.mean(rmses)):.4f} | params: {trial.params}")
    return float(np.mean(rmses))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print("Best RMSE:", study.best_value)
print("Best params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

[I 2026-05-27 18:13:28,954] A new study created in memory with name: no-name-7f0e106f-6bb9-4eef-ab34-e7118e8f0ef3
[I 2026-05-27 18:13:30,053] Trial 0 finished with value: 0.14201341050814756 and parameters: {'n_estimators': 327, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.14201341050814756.


Trial 0 | RMSE: 0.1420 | params: {'n_estimators': 327, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'log2'}


[I 2026-05-27 18:13:30,953] Trial 1 finished with value: 0.1444451000780719 and parameters: {'n_estimators': 242, 'max_depth': 6, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.14201341050814756.


Trial 1 | RMSE: 0.1444 | params: {'n_estimators': 242, 'max_depth': 6, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 'sqrt'}


[I 2026-05-27 18:13:32,105] Trial 2 finished with value: 0.14140824388910378 and parameters: {'n_estimators': 334, 'max_depth': 14, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 2 with value: 0.14140824388910378.


...

Trial 17 | RMSE: 0.1489 | params: {'n_estimators': 289, 'max_depth': 5, 'min_samples_split': 9, 'min_samples_leaf': 1, 'max_features': 'log2'}


[I 2026-05-27 18:13:48,417] Trial 18 finished with value: 0.1439417376695699 and parameters: {'n_estimators': 402, 'max_depth': 16, 'min_samples_split': 7, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 15 with value: 0.14126858325075364.


Trial 18 | RMSE: 0.1439 | params: {'n_estimators': 402, 'max_depth': 16, 'min_samples_split': 7, 'min_samples_leaf': 5, 'max_features': 'sqrt'}


[I 2026-05-27 18:13:49,306] Trial 19 finished with value: 0.1417525402278462 and parameters: {'n_estimators': 256, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 15 with value: 0.14126858325075364.


Trial 19 | RMSE: 0.1418 | params: {'n_estimators': 256, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2'}
Best RMSE: 0.14126858325075364
Best params:
  n_estimators: 282
  max_depth: 17
  min_samples_split: 9
  min_samples_leaf: 1
  max_features: log2


# 7. Final Model Training
Train the final Random Forest with best Optuna parameters on the training set, evaluate on holdout, retrain on full data, and save artifacts.

In [7]:
final_params = dict(best_params)
final_params.update({
    "random_state": 42,
    "n_jobs": -1
})

final_model = RandomForestRegressor(**final_params)
final_model.fit(X_train, y_train)

test_pred = final_model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, test_pred)))
mae = float(mean_absolute_error(y_test, test_pred))
mape = float(np.mean(np.abs((y_test - test_pred) / y_test.replace(0, np.nan))) * 100)

print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.4f}%")

print(f"Retraining on full dataset...")
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])
final_model.fit(X_full, y_full)

# Auto-save final retrained model and artifacts to local artifacts dir
model_path = ARTIFACTS_DIR / f"{safe_ticker}_random_forest_model.pkl"
best_params_path = ARTIFACTS_DIR / f"{safe_ticker}_rf_best_params.json"
feature_cols_path = ARTIFACTS_DIR / f"{safe_ticker}_feature_columns.json"

joblib.dump(final_model, model_path)
preprocessor.save_artifacts()

with feature_cols_path.open("w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print("Saved artifacts:")
print(f"- Model: {model_path.resolve()}")
print(f"- Preprocessor scaler: {(ARTIFACTS_DIR / 'scaler.pkl').resolve()}")
print(f"- Feature columns: {feature_cols_path.resolve()}")
print(f"- Best params: {best_params_path.resolve()}")

RMSE: 0.372507
MAE:  0.299785
MAPE: 30.3005%
Retraining on full dataset...
Saved artifacts:
- Model: C:\Users\gaura\Desktop\FinSight\notebooks\models\random_forest\HDFCBANK_NS_random_forest_model.pkl
- Preprocessor scaler: C:\Users\gaura\Desktop\FinSight\notebooks\models\random_forest\scaler.pkl
- Feature columns: C:\Users\gaura\Desktop\FinSight\notebooks\models\random_forest\HDFCBANK_NS_feature_columns.json
- Best params: C:\Users\gaura\Desktop\FinSight\notebooks\models\random_forest\HDFCBANK_NS_rf_best_params.json


# 8. Feature Importance
Visualize the top 15 most important predictors from the trained Random Forest model.

In [8]:
importance = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

fig_imp = go.Figure(
    go.Bar(
        x=importance.values[::-1],
        y=importance.index[::-1],
        orientation="h",
        marker_color="#2563eb"
    )
)
fig_imp.update_layout(
    title=f"{TICKER} Random Forest Top 15 Feature Importance",
    template="plotly_white",
    xaxis_title="Importance",
    yaxis_title="Feature",
    height=600
)
out_path = ARTIFACTS_DIR / f"{safe_ticker}_rf_feature_importance.html"
fig_imp.write_html(str(out_path))
print(f"Wrote feature importance to {out_path.resolve()}")

Wrote feature importance to C:\Users\gaura\Desktop\FinSight\notebooks\models\random_forest\HDFCBANK_NS_rf_feature_importance.html


# 9. Forecast Plot
Plot holdout-period actual vs predicted values and a 30-day future forecast using the latest feature row as a simple forward scenario.

In [9]:
future_steps = 30
last_row = X_test.iloc[[-1]] if len(X_test) > 0 else X_train.iloc[[-1]]
future_X = pd.concat([last_row] * future_steps, ignore_index=True)
future_pred = final_model.predict(future_X)

if isinstance(test_df.index, pd.DatetimeIndex):
    future_idx = pd.bdate_range(start=test_df.index[-1] + pd.tseries.offsets.BDay(1), periods=future_steps)
else:
    future_idx = pd.RangeIndex(start=len(test_df), stop=len(test_df) + future_steps)

y_actual = df.loc[test_df.index, "Close"]
pred_actual = preprocessor.inverse_transform(test_pred)
future_pred_actual = preprocessor.inverse_transform(future_pred)

fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=test_df.index, y=y_actual, mode="lines", name="Actual", line=dict(color="#111827", width=2)))
fig_fc.add_trace(go.Scatter(x=test_df.index, y=pred_actual, mode="lines", name="Predicted", line=dict(color="#2563eb", width=2)))
fig_fc.add_trace(go.Scatter(x=future_idx, y=future_pred_actual, mode="lines", name="30-Day Forecast", line=dict(color="#10b981", width=2, dash="dash")))
fig_fc.update_layout(
    title=f"{TICKER} Actual vs Predicted + 30-Day Forecast",
    template="plotly_white",
    xaxis_title="Date",
    yaxis_title="Close Price (₹)",
    height=600
)
out_path = ARTIFACTS_DIR / f"{safe_ticker}_rf_forecast.html"
fig_fc.write_html(str(out_path))
print(f"Wrote forecast plot to {out_path.resolve()}")

Wrote forecast plot to C:\Users\gaura\Desktop\FinSight\notebooks\models\random_forest\HDFCBANK_NS_rf_forecast.html


# 10. Save Artifacts
Persist the trained model, preprocessing artifacts, feature columns, and best hyperparameters under the local artifacts directory.

In [10]:
model_path = ARTIFACTS_DIR / f"{safe_ticker}_random_forest_model.pkl"
feature_cols_path = ARTIFACTS_DIR / f"{safe_ticker}_feature_columns.json"
best_params_path = ARTIFACTS_DIR / f"{safe_ticker}_rf_best_params.json"

joblib.dump(final_model, model_path)
preprocessor.save_artifacts()

with feature_cols_path.open("w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print("Saved artifacts:")
print(f"- Model: {model_path.resolve()}")
print(f"- Preprocessor scaler: {(ARTIFACTS_DIR / 'scaler.pkl').resolve()}")
print(f"- Feature columns: {feature_cols_path.resolve()}")
print(f"- Best params: {best_params_path.resolve()}")

Saved artifacts:
- Model: C:\Users\gaura\Desktop\FinSight\notebooks\models\random_forest\HDFCBANK_NS_random_forest_model.pkl
- Preprocessor scaler: C:\Users\gaura\Desktop\FinSight\notebooks\models\random_forest\scaler.pkl
- Feature columns: C:\Users\gaura\Desktop\FinSight\notebooks\models\random_forest\HDFCBANK_NS_feature_columns.json
- Best params: C:\Users\gaura\Desktop\FinSight\notebooks\models\random_forest\HDFCBANK_NS_rf_best_params.json
